# Llasa-8B - Voice Cloning

This notebook sets up inference for Llasa-8B, an 8-billion parameter Llama-based text-to-speech model. Because of the 8B parameter size, an A100 GPU with high VRAM is strictly required to run inference without offloading. Expected runtime is approximately 20 minutes.

In [ ]:
!nvidia-smi

In [ ]:
# Llasa has no package of its own (the "llasa" name on PyPI is unclaimed): it runs on
# transformers plus the authors' XCodec2 codec, pinned to the exact file the model card names.
!pip install "transformers>=4.45.2,<5" soundfile "xcodec2 @ https://files.pythonhosted.org/packages/80/69/5b99cc4de97f861d6b0a9acd20b4a22f44eabcc20051c0991b1cca479138/xcodec2-0.1.5.tar.gz#sha256=dc1a73b32090706e65fb73b2469411bc27bb72048677a23b430ab21ad325e45b"

In [ ]:
!huggingface-cli download HKUSTAudio/Llasa-8B

In [ ]:
from google.colab import files

print('Please upload your reference WAV file:')
uploaded = files.upload()
ref_wav_path = next(iter(uploaded))

# Llasa clones from the reference audio together with its exact transcript.
ref_text = 'Replace this with the exact words spoken in your reference clip.'

In [ ]:
import soundfile as sf
import torch
import torchaudio
from transformers import AutoModelForCausalLM, AutoTokenizer
from xcodec2.modeling_xcodec2 import XCodec2Model

print('Loading Llasa-8B model weights into VRAM...')
tokenizer = AutoTokenizer.from_pretrained('HKUSTAudio/Llasa-8B')
model = AutoModelForCausalLM.from_pretrained('HKUSTAudio/Llasa-8B')
model.eval().to('cuda')
codec = XCodec2Model.from_pretrained('HKUSTAudio/xcodec2')
codec.eval().cuda()

text = 'You are absolutely right. Your Claude Code session could sound like me.'

# XCodec2 only accepts 16 kHz mono audio.
prompt_wav, sr = torchaudio.load(ref_wav_path)
prompt_wav = torchaudio.functional.resample(prompt_wav.mean(dim=0, keepdim=True), sr, 16000)

print('Synthesizing speech from reference...')
with torch.no_grad():
    prompt_codes = codec.encode_code(input_waveform=prompt_wav)[0, 0, :].tolist()
    chat = [
        {'role': 'user', 'content': 'Convert the text to speech:'
            f'<|TEXT_UNDERSTANDING_START|>{ref_text}{text}<|TEXT_UNDERSTANDING_END|>'},
        {'role': 'assistant', 'content': '<|SPEECH_GENERATION_START|>'
            + ''.join(f'<|s_{code}|>' for code in prompt_codes)},
    ]
    input_ids = tokenizer.apply_chat_template(
        chat, tokenize=True, return_tensors='pt', continue_final_message=True
    ).to('cuda')
    outputs = model.generate(
        input_ids,
        max_length=2048,
        eos_token_id=tokenizer.convert_tokens_to_ids('<|SPEECH_GENERATION_END|>'),
        do_sample=True,
        top_p=1,
        temperature=0.8,
    )
    generated = tokenizer.batch_decode(outputs[0][input_ids.shape[1]:-1], skip_special_tokens=True)
    speech_ids = [int(tok[4:-2]) for tok in generated if tok.startswith('<|s_') and tok.endswith('|>')]
    audio = codec.decode_code(torch.tensor(speech_ids, device='cuda')[None, None, :])

sf.write('output.wav', audio[0, 0, :].cpu().numpy(), 16000)
print('Output saved to output.wav')

In [ ]:
from google.colab import files

files.download('output.wav')

### Next steps for the afterwords project

Running an 8B parameter model like Llasa natively on a 32GB Apple Silicon Mac leaves no memory for context buffering or other system tasks. To integrate this model into the `afterwords` project, you must set it up as an external backend. You can deploy this code in a Docker container on a remote A100 instance. Inside the `afterwords` repository, define a `Llasa8BBackend(BackendBase)` class where `load()` verifies the API connection, `prepare_voice()` uploads the reference audio to your cloud instance to cache it, and `synthesize()` calls the remote API with the text payload.